# Adım 4: EDA
Bronze tablosundan verileri analiz et.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, min, max, avg, hour, dayofweek

spark = SparkSession.builder \
    .appName("EDA") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

df = spark.read.format("delta").load("/app/delta/taxi_bronze")

print(f"Toplam Kayıt Sayısı: {df.count()}")
print("\nŞema Özeti:")
df.printSchema()

print("\nEksik Değer Analizi (Null Counts):")
null_counts = df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in df.columns])
null_counts.show()

print("\nÜcret ve Mesafe İstatistikleri:")
df.select("fare_amount", "trip_distance") \
    .summary("min", "25%", "50%", "75%", "max", "mean") \
    .show()

print("\nSaatlik Yolculuk Dağılımı (İlk 24 satır):")
df.withColumn("pickup_time", col("tpep_pickup_datetime").cast("timestamp")) \
    .withColumn("hour", hour("pickup_time")) \
    .groupBy("hour") \
    .count() \
    .orderBy("hour") \
    .show(24)

# 5. Aykırı Değer Kontrolü
invalid_fares = df.filter(col("fare_amount") <= 0).count()
print(f"\nÜcreti 0 veya negatif olan hatalı kayıt sayısı: {invalid_fares}")


Toplam Kayıt Sayısı: 1000000

Şema Özeti:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- kullanici_id: string (nullable = true)
 |-- olay_tipi: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- ilgili_id: string (nullable = true)


Eksik Değer Analizi (Null Counts):
+--------+--------------------+---------------------+---------------+-------------+-----------+------------+------------+------------+---------+---------+---------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|PULocationID|DOLocationID|kullanici_id|olay_tipi|timestamp|ilgili_id|
+--------+--------------------+-----------------